In [1]:
%%writefile /kaggle/working/data.py
"""Dataset and transforms. One canonical definition shared by every model notebook."""
import os
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms as T

IN   = '/kaggle/input/notebooks/rayyanshuda/02-split-and-preprocess'
CSV  = f'{IN}/split_v1.csv'
IMGS = f'{IN}/resized'

MEAN, STD, SIZE = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 160

# Augmentation on train only. No colour jitter here on purpose — it is the
# intervention in the shortcut experiment and must not contaminate the baseline.
train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([
    T.Resize(SIZE),
    T.CenterCrop(SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


class WildfireDataset(Dataset):
    """Reads split_v1.csv; labels fire=1.0, nofire=0.0."""

    def __init__(self, csv_path, img_dir, split, transform):
        df = pd.read_csv(csv_path)
        df = df[df.split == split].reset_index(drop=True)
        self.files      = df.out_name.tolist()
        self.labels     = (df.cls == 'fire').astype('float32').tolist()
        self.img_dir    = img_dir
        self.transform  = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        path = os.path.join(self.img_dir, self.files[i])
        img  = Image.open(path).convert('RGB')      # 12 images are RGBA
        img  = self.transform(img)
        return img, torch.tensor(self.labels[i], dtype=torch.float32)

Writing /kaggle/working/data.py


In [2]:
%%writefile /kaggle/working/train.py
"""Training loop and prediction. Every model arm runs this identical code —
that is what makes the benchmark a fair comparison rather than a claim."""
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, roc_curve, log_loss

from data import WildfireDataset, train_tf, eval_tf, CSV, IMGS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def predict(model, loader):
    """Return (y_true, y_score) as numpy. The only torch-aware piece."""
    model.eval()
    ys, ss = [], []
    with torch.no_grad():
        for xb, yb in loader:
            ss.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ss)


def pick_threshold(y_true, y_score):
    """Youden's J: maximise (sensitivity + specificity - 1). Chosen on VAL only."""
    fpr, tpr, thr = roc_curve(y_true, y_score)
    t = float(thr[np.argmax(tpr - fpr)])
    return t if np.isfinite(t) else 0.5


def train_model(make_model, seed, epochs=40, lr=1e-3, patience=8, bs=32,
                tag='model', freeze_epochs=0, train_idx=None, val_idx=None):
    """train_idx: explicit list of training indices (stratified subsets for the
    data-efficiency experiment). val_idx: only for fast dry runs — real runs use
    the FULL val set at every training size, or the sizes aren't comparable."""
    torch.manual_seed(seed); np.random.seed(seed)

    train_ds = WildfireDataset(CSV, IMGS, 'train', train_tf)
    val_ds   = WildfireDataset(CSV, IMGS, 'val',   eval_tf)
    if train_idx is not None:
        train_ds = Subset(train_ds, list(train_idx))
    if val_idx is not None:
        val_ds = Subset(val_ds, list(val_idx))

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=len(train_ds) > bs)
    val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)

    model     = make_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()

    def build_optim(phase):
        if phase == 'frozen':
            for p in model.backbone_parameters(): p.requires_grad = False
            opt, T = torch.optim.Adam(model.head_parameters(), lr=lr), freeze_epochs
        elif phase == 'unfrozen':
            for p in model.backbone_parameters(): p.requires_grad = True
            opt = torch.optim.Adam([
                {'params': model.backbone_parameters(), 'lr': lr / 100},
                {'params': model.head_parameters(),     'lr': lr / 10},
            ])
            T = epochs - freeze_epochs
        else:
            opt, T = torch.optim.Adam(model.parameters(), lr=lr), epochs
        return opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(T, 1))

    optimizer, scheduler = build_optim('frozen' if freeze_epochs > 0 else 'single')
    best_auc, best_state, best_epoch, bad = -1, None, -1, 0
    history = []

    for epoch in range(epochs):
        if freeze_epochs > 0 and epoch == freeze_epochs:
            optimizer, scheduler = build_optim('unfrozen')
            print('  --- backbone unfrozen ---')

        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * len(yb); n += len(yb)
        scheduler.step()

        yt, ys   = predict(model, val_dl)
        val_loss = log_loss(yt, np.clip(ys, 1e-7, 1 - 1e-7), labels=[0, 1])
        val_auc  = roc_auc_score(yt, ys)
        history.append({'epoch': epoch, 'train_loss': run_loss / n,
                        'val_loss': val_loss, 'val_auc': val_auc,
                        'lr': scheduler.get_last_lr()[0]})
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f'  ep {epoch:3d}  train {run_loss/n:.4f}  val {val_loss:.4f}  auc {val_auc:.4f}')

        if val_auc > best_auc:
            best_auc, best_epoch, bad = val_auc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                print(f'  early stop at {epoch}; best epoch was {best_epoch}')
                break

    model.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/{tag}_seed{seed}.pt')
    pd.DataFrame(history).to_csv(f'/kaggle/working/{tag}_seed{seed}_history.csv', index=False)
    yt_v, ys_v = predict(model, val_dl)
    return model, pd.DataFrame(history), best_auc, best_epoch, pick_threshold(yt_v, ys_v)

Writing /kaggle/working/train.py


In [3]:
%%writefile /kaggle/working/models.py
"""The two architectures under comparison."""
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


class ModelA(nn.Module):
    """Custom scratch CNN. 93,601 params."""

    def __init__(self, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x).squeeze(1)


class ModelB(nn.Module):
    """EfficientNet-B0. 4,008,829 params. pretrained=False is the control arm."""

    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.net = efficientnet_b0(weights=weights)
        self.net.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)

    def backbone_parameters(self):
        return self.net.features.parameters()

    def head_parameters(self):
        return self.net.classifier.parameters()

Writing /kaggle/working/models.py


In [4]:
import os
for r, d, f in os.walk('/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines'):
    print(r, '->', f)

/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines -> ['evaluate.py', '__results__.html', '__notebook__.ipynb', '__output__.json', 'results.csv', 'custom.css']
/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines/__pycache__ -> ['evaluate.cpython-312.pyc']
/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines/__results___files -> ['__results___5_0.png']


In [5]:
import sys
sys.path.insert(0, '/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines')
from evaluate import compute_metrics, evaluate_all

In [6]:
import data, train, evaluate, models
import importlib
for m in (data, models, train, evaluate):
    importlib.reload(m)

from models import ModelA, ModelB

In [7]:
import sys, importlib
sys.path.insert(0, '/kaggle/working')
importlib.invalidate_caches()

import data, train, evaluate
for m in (data, train, evaluate):
    importlib.reload(m)

from data     import WildfireDataset, train_tf, eval_tf, CSV, IMGS, SIZE
from train    import predict, pick_threshold, train_model, DEVICE
from evaluate import compute_metrics, evaluate_all

print('device:', DEVICE)
print(len(WildfireDataset(CSV, IMGS, 'train', train_tf)),
      len(WildfireDataset(CSV, IMGS, 'val',   eval_tf)),
      len(WildfireDataset(CSV, IMGS, 'test',  eval_tf)))    # expect 1803 420 476

device: cuda
1803 420 476


In [8]:
import numpy as np, pandas as pd, torch, os

SIZES, FULL, EPOCH_CAP, BASE = [100, 250, 500, 1000], 1803, 120, 40

def epochs_for(n):
    return int(min(EPOCH_CAP, max(BASE, round(BASE * FULL / n))))

train_meta = pd.read_csv(CSV)
train_meta = train_meta[train_meta.split == 'train'].reset_index(drop=True)
train_meta['stratum'] = train_meta.src + '_' + train_meta.cls
print(train_meta.stratum.value_counts().to_string())

def nested_subsets(meta, sizes, seed):
    """Proportional prefixes of a per-stratum shuffle -> stratified AND nested."""
    rng = np.random.RandomState(1000 + seed)
    order = {}
    for s, grp in meta.groupby('stratum'):
        idx = grp.index.to_numpy().copy()
        rng.shuffle(idx)
        order[s] = idx
    N, out = len(meta), {}
    for n in sizes:
        take = []
        for s, idx in order.items():
            take.extend(idx[:max(1, round(len(idx) * n / N))])
        out[n] = sorted(take)
    return out

sub = nested_subsets(train_meta, SIZES, seed=0)
for n in SIZES:
    got = train_meta.loc[sub[n]]
    print(f'target {n:5d} -> actual {len(got):5d} | {got.stratum.value_counts().to_dict()}')
assert set(sub[100]).issubset(set(sub[250])), 'subsets are not nested'

stratum
unsplash_nofire    1014
flickr_fire         688
flickr_nofire        73
unsplash_fire        28
target   100 -> actual   100 | {'unsplash_nofire': 56, 'flickr_fire': 38, 'flickr_nofire': 4, 'unsplash_fire': 2}
target   250 -> actual   250 | {'unsplash_nofire': 141, 'flickr_fire': 95, 'flickr_nofire': 10, 'unsplash_fire': 4}
target   500 -> actual   500 | {'unsplash_nofire': 281, 'flickr_fire': 191, 'flickr_nofire': 20, 'unsplash_fire': 8}
target  1000 -> actual  1000 | {'unsplash_nofire': 562, 'flickr_fire': 382, 'flickr_nofire': 40, 'unsplash_fire': 16}


In [9]:
import os
import pandas as pd
from torch.utils.data import DataLoader

DRY = False                                    # <<< set False for the real run

ARM = 'b0_pretrained'                               # 'model_a' | 'b0_scratch' | 'b0_pretrained'

ARM_SPEC = {
    'model_a':       (lambda: ModelA(),                 0),
    'b0_scratch':    (lambda: ModelB(pretrained=False), 0),
    'b0_pretrained': (lambda: ModelB(pretrained=True),  3),
}
maker, fz = ARM_SPEC[ARM]

RUN_SIZES = [100] if DRY else SIZES
RUN_SEEDS = [0]   if DRY else [0, 1, 2]
OUT       = f'/kaggle/working/dataeff_{"DRY_" if DRY else ""}{ARM}.csv'

test_dl   = DataLoader(WildfireDataset(CSV, IMGS, 'test', eval_tf), batch_size=64,
                       shuffle=False, num_workers=2, pin_memory=True)
test_meta = pd.read_csv(CSV)
test_meta = test_meta[test_meta.split == 'test'].reset_index(drop=True)

for seed in RUN_SEEDS:
    subs = nested_subsets(train_meta, SIZES, seed)      # always build the full family
    for n in RUN_SIZES:
        idx, ep = subs[n], epochs_for(n)
        if DRY:
            ep = 10                      # <<< dry run: just prove the plumbing
        tag = f'{ARM}_n{n}_seed{seed}'
        print(f'\n===== {tag} | {len(idx)} images | {ep} epochs =====')

        model, hist, bauc, bep, thr = train_model(
            maker, seed, epochs=ep, tag=tag, freeze_epochs=fz, train_idx=idx)

        yt, ys = predict(model, test_dl)
        rows = evaluate_all(yt, ys, test_meta.src.values, tag, threshold=thr)
        rows['arm'], rows['n_train'], rows['seed'] = ARM, len(idx), seed
        rows['epochs'], rows['best_epoch'], rows['best_val_auc'] = ep, bep, bauc

        rows.to_csv(OUT, mode='a', header=not os.path.exists(OUT), index=False)
        print(rows.loc[rows['slice'] == 'source-matched (flickr)',
                       ['n_train', 'seed', 'roc_auc', 'accuracy']].to_string(index=False))

print('\n===== summary =====')
df = pd.read_csv(OUT)
for sl in ['source-matched (flickr)', 'full test']:
    print(f'\n--- {sl} ---')
    print(df[df['slice'] == sl].groupby('n_train')[['roc_auc', 'accuracy', 'specificity']]
          .agg(['mean', 'std']).round(4).to_string())


===== b0_pretrained_n100_seed0 | 100 images | 120 epochs =====
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 167MB/s]


  ep   0  train 0.7071  val 0.7133  auc 0.4059
  --- backbone unfrozen ---
  ep  10  train 0.5995  val 0.6487  auc 0.6772
  ep  20  train 0.5176  val 0.6165  auc 0.7645
  ep  30  train 0.4552  val 0.5812  auc 0.8183
  ep  40  train 0.3859  val 0.5496  auc 0.8458
  ep  50  train 0.3942  val 0.5243  auc 0.8601
  ep  60  train 0.3146  val 0.5067  auc 0.8701
  ep  70  train 0.2880  val 0.4911  auc 0.8760
  ep  80  train 0.2842  val 0.4810  auc 0.8808
  ep  90  train 0.2392  val 0.4746  auc 0.8840
  ep 100  train 0.2657  val 0.4734  auc 0.8819
  early stop at 102; best epoch was 94
 n_train  seed  roc_auc  accuracy
     100     0   0.8649    0.8057

===== b0_pretrained_n250_seed0 | 250 images | 120 epochs =====
  ep   0  train 0.7021  val 0.6782  auc 0.5628
  --- backbone unfrozen ---
  ep  10  train 0.5088  val 0.5676  auc 0.8334
  ep  20  train 0.3964  val 0.4906  auc 0.8832
  ep  30  train 0.2939  val 0.4296  auc 0.9060
  ep  40  train 0.2409  val 0.3831  auc 0.9207
  ep  50  train 0.195